# task_20 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

fields = pd.read_csv(base / "fields.csv")
plantings = pd.read_csv(base / "plantings.csv")
harvests = pd.read_csv(base / "harvests.csv")
pests = pd.read_csv(base / "pests.csv")

harvests["harvest_date"] = pd.to_datetime(harvests["harvest_date"])
plantings["target_harvest_date"] = pd.to_datetime(plantings["target_harvest_date"])
pests["week_start"] = pd.to_datetime(pests["week_start"])

yield_df = fields.merge(harvests, on="field_id")
yield_df["yield_per_ha"] = yield_df["yield_tons"] / yield_df["area_ha"]
q1 = str(yield_df.groupby("crop")["yield_per_ha"].mean().sort_values(ascending=False).index[0])

crop_avg = yield_df.groupby("crop")["yield_per_ha"].mean().rename("crop_avg")
shortfall = yield_df.join(crop_avg, on="crop")
shortfall["yield_shortfall"] = shortfall["crop_avg"] - shortfall["yield_per_ha"]
q2 = str(shortfall.sort_values(["yield_shortfall", "field_id"], ascending=[False, True]).iloc[0]["field_id"])

corn = fields.merge(plantings, on="field_id").merge(harvests, on="field_id")
q3 = str(int(corn[(corn["crop"] == "Corn") & (corn["harvest_date"] > corn["target_harvest_date"])]["irrigation_mm"].sum()))

q4 = str(pests.groupby("week_start")["pest_incidents"].sum().sort_values(ascending=False).index[0].date().isoformat())

late_df = fields.merge(harvests, on="field_id").merge(plantings[["field_id", "target_harvest_date"]], on="field_id")
late_df["yield_per_ha"] = late_df["yield_tons"] / late_df["area_ha"]
crop_median = late_df.groupby("crop")["yield_per_ha"].median().rename("crop_median")
late_df = late_df.join(crop_median, on="crop")
condition = (late_df["harvest_date"] > late_df["target_harvest_date"]) & (late_df["yield_per_ha"] < late_df["crop_median"])
q5 = f"{(100 * condition.mean()):.1f}%"


Q1: Which crop has the highest average yield per hectare?

In [ ]:
print(q1)


Q2: Which field_id has the greatest shortfall in yield per hectare versus the average yield per hectare of its own crop?

In [ ]:
print(q2)


Q3: What is the total irrigation_mm across Corn fields that were harvested after their target_harvest_date?

In [ ]:
print(q3)


Q4: Which week_start has the highest total pest_incidents?

In [ ]:
print(q4)


Q5: What percentage of fields were harvested after target_harvest_date and also had yield per hectare below the median yield per hectare of their crop, rounded to 1 decimal place?

In [ ]:
print(q5)
